# Notebook 06: Multi-Agent Orchestration with LangGraph

**Phase 7: User Story 5 - Multi-Agent Orchestration**  
**Tasks**: T068 (Orchestration Notebook), T069 (State Transition Diagram)

## Learning Objectives

1. Understand LangGraph state machine patterns for agent coordination
2. Execute sequential multi-agent workflows with conditional routing
3. Visualize state evolution across agent nodes
4. Handle errors gracefully with error routing
5. Track execution timing and costs across the workflow
6. Analyze workflow performance metrics

## Overview

This notebook demonstrates the complete **multi-agent loan underwriting workflow** using LangGraph for orchestration:

- **Document Agent** → Extracts data from loan application documents
- **Risk Agent** → Queries credit bureau, calculates financial ratios, performs risk analysis
- **Compliance Agent** → Checks policy compliance using RAG retrieval
- **Decision Agent** → Makes final lending decision with rate calculation
- **Error Handler** → Handles workflow failures gracefully

Each agent updates a shared `ApplicationState`, enabling transparent, auditable decision-making.

## 1. Environment Setup and Imports

In [1]:
import sys
import os
from pathlib import Path
import json
from datetime import datetime
import time

# Add src to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

# Import orchestrator
from orchestrator import (
    create_initial_state,
    create_workflow,
    run_workflow,
    get_state_summary,
    is_workflow_complete,
    has_errors
)

# Import models for validation
from models import LoanApplication, LendingDecision

# Visualization imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print("✅ Environment setup complete")
print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version.split()[0]}")

✅ Environment setup complete
📁 Project root: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan
🐍 Python version: 3.11.9


## 2. State Transition Diagram (T069)

The LangGraph workflow coordinates four agent nodes with conditional error routing:

```mermaid
graph TD
    START([START]) --> document[📄 Document Agent]
    
    document -->|success| risk[📊 Risk Agent]
    document -->|error| error_handler[🚨 Error Handler]
    
    risk -->|success| compliance[✅ Compliance Agent]
    risk -->|error| error_handler
    
    compliance -->|success| decision[⚖️ Decision Agent]
    compliance -->|error| error_handler
    
    decision -->|success| complete([✓ COMPLETE])
    decision -->|error| error_handler
    
    error_handler --> END([END])
    complete --> END
    
    style START fill:#e1f5e1
    style document fill:#fff3cd
    style risk fill:#cfe2ff
    style compliance fill:#d1ecf1
    style decision fill:#f8d7da
    style error_handler fill:#f8d7da
    style complete fill:#d4edda
    style END fill:#e1e5e8
```

**Workflow Logic:**
- **Sequential Execution**: document → risk → compliance → decision
- **Error Routing**: Any agent failure routes to error_handler (terminal state)
- **State Accumulation**: Each agent adds to shared ApplicationState
- **Conditional Edges**: `should_continue()` function checks errors after each node

## 3. Create Sample Loan Application

In [2]:
# Sample loan application - strong profile
sample_application = {
    "application_id": "APP-2025-001",
    "first_name": "Alice",
    "last_name": "Johnson",
    "ssn": "123-45-6789",  # Maps to excellent credit (780) in mock DB
    "email": "alice.johnson@example.com",
    "phone": "+1-555-0123",
    "date_of_birth": "1985-03-15",
    "annual_income": 120000,
    "employment_status": "employed",
    "employer_name": "Tech Corp",
    "years_at_current_job": 5,
    "requested_amount": 300000,
    "loan_purpose": "home_purchase",
    "property_address": "123 Main St, San Francisco, CA 94102",
    "property_value": 400000,
    "down_payment": 100000,
    "document_paths": [
        str(project_root / "data" / "applications" / "paystub_sample.pdf")
    ]
}

print("📋 Sample Application Created")
print(f"   Application ID: {sample_application['application_id']}")
print(f"   Applicant: {sample_application['first_name']} {sample_application['last_name']}")
print(f"   Requested Amount: ${sample_application['requested_amount']:,}")
print(f"   Property Value: ${sample_application['property_value']:,}")
print(f"   LTV: {(sample_application['requested_amount'] / sample_application['property_value'] * 100):.1f}%")
print(f"   Documents: {len(sample_application['document_paths'])} file(s)")


📋 Sample Application Created
   Application ID: APP-2025-001
   Applicant: Alice Johnson
   Requested Amount: $300,000
   Property Value: $400,000
   LTV: 75.0%
   Documents: 1 file(s)


## 4. Execute Workflow - Happy Path

In [3]:
# Quick verification - test workflow creation
try:
    test_workflow = create_workflow()
    print("✅ Workflow created successfully")
    print(f"   Workflow type: {type(test_workflow)}")
    print(f"   Ready to process applications")
except Exception as e:
    print(f"❌ Error creating workflow: {e}")
    import traceback
    traceback.print_exc()

✅ Workflow created successfully
   Workflow type: <class 'langgraph.graph.state.CompiledStateGraph'>
   Ready to process applications


In [4]:
# Run complete workflow
print("🚀 Starting multi-agent workflow...\n")

final_state = run_workflow(sample_application)

print("\n" + "="*60)
print("📊 WORKFLOW EXECUTION COMPLETE")
print("="*60)

🚀 Starting multi-agent workflow...



INFO:orchestrator:📄 Document Agent starting for application APP-2025-001
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
INFO:orchestrator:  Analyzing document: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/data/applications/paystub_sample.pdf
INFO:agents.document_agent:Analyzing document: paystub_sample.pdf (type: pay_stub)
INFO:agents.document_agent:Calling Document Intelligence with model: prebuilt-read
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read:analyze?stringIndexType=unicodeCodePoint&api-version=2023-07-31'
Request method: 'POST'
Request headers:
    'Content-Length': '2078'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/json'
    'x-ms-client-request-id': '36b4bc82-d0c7-11f0-b


📊 WORKFLOW EXECUTION COMPLETE


## 5. Visualize State Evolution

In [5]:
# Display state summary
summary = get_state_summary(final_state)

print("📊 WORKFLOW SUMMARY")
print(f"   Current Agent: {summary['current_agent']}")
print(f"   Total Duration: {summary['total_duration']:.2f}s")
print(f"   Total Tokens: {summary['total_tokens']:,}")
print(f"   Total Cost: ${summary['total_cost']:.4f}")
print(f"   Errors: {summary['error_count']}")
print()

# Show state fields populated
print("📋 STATE FIELDS POPULATED:")
state_fields = [
    ('Application ID', final_state.get('application_id')),
    ('Started At', final_state.get('started_at')),
    ('Extracted Documents', len(final_state.get('extracted_documents', [])) if final_state.get('extracted_documents') else 0),
    ('Credit Report', '✓' if final_state.get('credit_report') else '✗'),
    ('Risk Assessment', '✓' if final_state.get('risk_assessment') else '✗'),
    ('Compliance Report', '✓' if final_state.get('compliance_report') else '✗'),
    ('Lending Decision', '✓' if final_state.get('lending_decision') else '✗'),
]

for field_name, value in state_fields:
    print(f"   {field_name}: {value}")

📊 WORKFLOW SUMMARY
   Current Agent: complete
   Total Duration: 12.89s
   Total Tokens: 6,500
   Total Cost: $0.0620
   Errors: 0

📋 STATE FIELDS POPULATED:
   Application ID: APP-2025-001
   Started At: 2025-12-04 04:10:48.126032
   Extracted Documents: 1
   Credit Report: ✓
   Risk Assessment: ✓
   Compliance Report: ✓
   Lending Decision: ✓


## 6. Display Final Lending Decision

In [6]:
# Extract and display lending decision
if final_state.get('lending_decision'):
    decision = final_state['lending_decision']
    
    print("⚖️ LENDING DECISION")
    print("="*60)
    print(f"   Decision: {decision['decision'].upper()}")
    print(f"   Interest Rate: {decision.get('interest_rate', 'N/A')}%")
    print(f"   Monthly Payment: ${decision.get('monthly_payment', 0):,.2f}")
    print(f"   Confidence: {decision.get('confidence', 0):.2%}")
    print()
    print(f"📝 Explanation:")
    print(f"   {decision.get('explanation', 'No explanation available')}")
    print()
    
    # Display risk factors
    if decision.get('risk_factors'):
        print("⚠️ Risk Factors:")
        for factor in decision['risk_factors']:
            print(f"   • {factor}")
        print()
    
    # Display conditions (if conditional approval)
    if decision.get('conditions'):
        print("📋 Conditions:")
        for condition in decision['conditions']:
            print(f"   • {condition}")
else:
    print("❌ No lending decision available - workflow may have encountered errors")

⚖️ LENDING DECISION
   Decision: CONDITIONAL_APPROVAL
   Interest Rate: 6.75%
   Monthly Payment: $1,945.79
   Confidence: 0.00%

📝 Explanation:
   No explanation available

📋 Conditions:
   • Obtain proof of stable employment or income verification for at least the last 2 years.
   • Require PMI due to the LTV being at 75%.


## 7. Execution Time Analysis

In [7]:
# Visualize execution times per agent
execution_times = final_state.get('execution_times', {})

if execution_times:
    agents = list(execution_times.keys())
    times = list(execution_times.values())
    
    fig = go.Figure(data=[
        go.Bar(
            x=agents,
            y=times,
            text=[f"{t:.2f}s" for t in times],
            textposition='auto',
            marker=dict(
                color=times,
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title="Time (s)")
            )
        )
    ])
    
    fig.update_layout(
        title="Agent Execution Times",
        xaxis_title="Agent",
        yaxis_title="Time (seconds)",
        height=400,
        showlegend=False
    )
    
    fig.show()
    
    # Print timing table
    print("\n⏱️ EXECUTION TIMING BREAKDOWN")
    print("-" * 40)
    for agent, time_val in execution_times.items():
        percentage = (time_val / sum(times)) * 100
        print(f"   {agent:20s}: {time_val:6.2f}s ({percentage:5.1f}%)")
    print("-" * 40)
    print(f"   {'TOTAL':20s}: {sum(times):6.2f}s (100.0%)")
else:
    print("⚠️ No execution timing data available")


⏱️ EXECUTION TIMING BREAKDOWN
----------------------------------------
   document            :   2.69s ( 20.9%)
   risk                :   3.50s ( 27.2%)
   compliance          :   4.81s ( 37.3%)
   decision            :   1.89s ( 14.6%)
----------------------------------------
   TOTAL               :  12.89s (100.0%)


## 8. Cost Breakdown Analysis

In [8]:
# Display cost metrics
total_tokens = final_state.get('total_tokens_used', 0)
total_cost = final_state.get('total_cost_usd', 0)

print("💰 COST BREAKDOWN")
print("="*60)
print(f"   Total Tokens: {total_tokens:,}")
print(f"   Total Cost: ${total_cost:.4f}")
print()

# Estimate cost per agent (based on typical usage patterns)
if total_tokens > 0:
    # Approximate breakdown (actual may vary)
    agent_costs = {
        'Document Agent': 0.30,  # ~30% (Document Intelligence + normalization)
        'Risk Agent': 0.25,      # ~25% (GPT-4o risk analysis)
        'Compliance Agent': 0.30, # ~30% (RAG + GPT-4o compliance)
        'Decision Agent': 0.15   # ~15% (Final analysis + explanation)
    }
    
    print("📊 Estimated Cost Distribution:")
    for agent, percentage in agent_costs.items():
        estimated_cost = total_cost * percentage
        print(f"   {agent:25s}: ${estimated_cost:.4f} ({percentage*100:.0f}%)")
    print()
    
    # Cost efficiency note
    print("💡 Cost Efficiency Notes:")
    print("   • Document Intelligence: ~$0.002 per document page")
    print("   • GPT-4o calls: Variable based on input/output length")
    print("   • Typical workflow: ~$0.065 per application")
    print("   • Caching can reduce costs by ~50% for similar queries")
else:
    print("⚠️ No cost data available")

💰 COST BREAKDOWN
   Total Tokens: 6,500
   Total Cost: $0.0620

📊 Estimated Cost Distribution:
   Document Agent           : $0.0186 (30%)
   Risk Agent               : $0.0155 (25%)
   Compliance Agent         : $0.0186 (30%)
   Decision Agent           : $0.0093 (15%)

💡 Cost Efficiency Notes:
   • Document Intelligence: ~$0.002 per document page
   • GPT-4o calls: Variable based on input/output length
   • Typical workflow: ~$0.065 per application
   • Caching can reduce costs by ~50% for similar queries


## 9. Error Handling Scenario

Let's test the error routing by submitting an application with missing critical data.

In [9]:
# Create application with invalid/missing data to trigger error
error_application = {
    "application_id": "APP-ERROR-001",
    "first_name": "Bob",
    "last_name": "Error",
    "ssn": "999-99-9999",  # Invalid SSN - won't find credit report
    "requested_amount": 300000,
    "property_value": 400000,
    "document_paths": []  # No documents - will fail document extraction
}

print("🚨 Testing Error Handling\n")
print("   Invalid application with:")
print("   • No documents (will fail document agent)")
print("   • Invalid SSN (would fail risk agent if it got there)")
print()

error_state = run_workflow(error_application)

print("\n" + "="*60)
print("📊 ERROR WORKFLOW RESULT")
print("="*60)

# Check if workflow detected error
if has_errors(error_state):
    print(f"✓ Error routing worked correctly")
    print(f"   Current agent: {error_state.get('current_agent')}")
    print(f"   Errors encountered: {len(error_state.get('errors', []))}")
    print()
    print("⚠️ Error Details:")
    for idx, error in enumerate(error_state.get('errors', []), 1):
        print(f"   {idx}. {error}")
else:
    print("⚠️ No errors detected (unexpected)")

INFO:orchestrator:Creating initial state for application APP-ERROR-001
INFO:orchestrator:🚀 Starting workflow for application APP-ERROR-001
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → compliance → decision
INFO:orchestrator:   Error handling: Conditional routing after each node
INFO:orchestrator:📄 Document Agent starting for application APP-ERROR-001
INFO:orchestrator:🚀 Starting workflow for application APP-ERROR-001
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → compliance → decision
INFO:orchestrator:   Error handling: Conditional routing after each node
INFO:orchestrator:📄 Document Agent starting for application APP-ERROR-001
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
ERROR:orchestrator:❌ Document Agent error: No documents found in loan application
ERROR

🚨 Testing Error Handling

   Invalid application with:
   • No documents (will fail document agent)
   • Invalid SSN (would fail risk agent if it got there)


📊 ERROR WORKFLOW RESULT
✓ Error routing worked correctly
   Current agent: error
   Errors encountered: 1

⚠️ Error Details:
   1. document_agent: No documents found in loan application


## 10. Workflow Comparison - Multiple Applications

Let's process multiple applications and compare their outcomes.

In [10]:
# Define multiple test applications with varying profiles
test_applications = [
    {
        "application_id": "APP-EXCELLENT-001",
        "first_name": "Alice", "last_name": "Excellent",
        "ssn": "123-45-6789",  # Excellent credit (780)
        "requested_amount": 300000,
        "property_value": 400000,
        "document_paths": [str(project_root / "data" / "applications" / "paystub_sample.pdf")]
    },
    {
        "application_id": "APP-GOOD-002",
        "first_name": "Bob", "last_name": "Good",
        "ssn": "234-56-7890",  # Good credit (720)
        "requested_amount": 350000,
        "property_value": 450000,
        "document_paths": [str(project_root / "data" / "applications" / "paystub_sample.pdf")]
    },
    {
        "application_id": "APP-FAIR-003",
        "first_name": "Carol", "last_name": "Fair",
        "ssn": "345-67-8901",  # Fair credit (670)
        "requested_amount": 250000,
        "property_value": 350000,
        "document_paths": [str(project_root / "data" / "applications" / "paystub_sample.pdf")]
    },
    {
        "application_id": "APP-POOR-004",
        "first_name": "David", "last_name": "Poor",
        "ssn": "456-78-9012",  # Poor credit (590)
        "requested_amount": 400000,
        "property_value": 450000,
        "document_paths": [str(project_root / "data" / "applications" / "paystub_sample.pdf")]
    }
]

# Process all applications
print("🔄 Processing multiple applications...\n")

results = []
for app in test_applications:
    print(f"Processing {app['application_id']}...")
    state = run_workflow(app)
    
    result = {
        'app_id': app['application_id'],
        'name': f"{app['first_name']} {app['last_name']}",
        'ssn': app['ssn'],
        'decision': state.get('lending_decision', {}).get('decision', 'N/A'),
        'interest_rate': state.get('lending_decision', {}).get('interest_rate', 0),
        'duration': sum(state.get('execution_times', {}).values()),
        'cost': state.get('total_cost_usd', 0),
        'errors': len(state.get('errors', []))
    }
    results.append(result)
    print(f"   ✓ Complete - Decision: {result['decision']}\n")

print("="*60)
print("📊 WORKFLOW COMPARISON COMPLETE")
print("="*60)


INFO:orchestrator:Creating initial state for application APP-EXCELLENT-001
INFO:orchestrator:🚀 Starting workflow for application APP-EXCELLENT-001
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → compliance → decision
INFO:orchestrator:   Error handling: Conditional routing after each node
INFO:orchestrator:📄 Document Agent starting for application APP-EXCELLENT-001
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
INFO:orchestrator:  Analyzing document: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/data/applications/paystub_sample.pdf
INFO:agents.document_agent:Analyzing document: paystub_sample.pdf (type: pay_stub)
INFO:orchestrator:🚀 Starting workflow for application APP-EXCELLENT-001
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → 

🔄 Processing multiple applications...

Processing APP-EXCELLENT-001...


INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/4b3fa263-4cd5-4f33-995e-f228ec364b75?api-version=2023-07-31'
    'x-envoy-upstream-service-time': '43'
    'apim-request-id': '4b3fa263-4cd5-4f33-995e-f228ec364b75'
    'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload'
    'x-content-type-options': 'nosniff'
    'x-ms-region': 'East US 2'
    'Date': 'Thu, 04 Dec 2025 04:11:02 GMT'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/4b3fa263-4cd5-4f33-995e-f228ec364b75?api-version=2023-07-31'
Request method: 'GET'
Request headers:
    'x-ms-client-request-id': '3f184934-d0c7-11f0-b983-56107ba8ade

   ✓ Complete - Decision: conditional_approval

Processing APP-GOOD-002...


INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/c914ef3d-23ae-4258-af2f-42227d54c007?api-version=2023-07-31'
    'x-envoy-upstream-service-time': '46'
    'apim-request-id': 'c914ef3d-23ae-4258-af2f-42227d54c007'
    'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload'
    'x-content-type-options': 'nosniff'
    'x-ms-region': 'East US 2'
    'Date': 'Thu, 04 Dec 2025 04:11:14 GMT'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/c914ef3d-23ae-4258-af2f-42227d54c007?api-version=2023-07-31'
Request method: 'GET'
Request headers:
    'x-ms-client-request-id': '468c00ac-d0c7-11f0-b983-56107ba8ade

   ✓ Complete - Decision: conditional_approval

Processing APP-FAIR-003...


INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/15b78430-e33e-4451-93d4-32cdf8accf8a?api-version=2023-07-31'
    'x-envoy-upstream-service-time': '46'
    'apim-request-id': '15b78430-e33e-4451-93d4-32cdf8accf8a'
    'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload'
    'x-content-type-options': 'nosniff'
    'x-ms-region': 'East US 2'
    'Date': 'Thu, 04 Dec 2025 04:11:33 GMT'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/15b78430-e33e-4451-93d4-32cdf8accf8a?api-version=2023-07-31'
Request method: 'GET'
Request headers:
    'x-ms-client-request-id': '518d6c3e-d0c7-11f0-b983-56107ba8ade

   ✓ Complete - Decision: conditional_approval

Processing APP-POOR-004...


INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/a26d2163-b3ce-4167-b412-73f5bf9746d9?api-version=2023-07-31'
    'x-envoy-upstream-service-time': '47'
    'apim-request-id': 'a26d2163-b3ce-4167-b412-73f5bf9746d9'
    'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload'
    'x-content-type-options': 'nosniff'
    'x-ms-region': 'East US 2'
    'Date': 'Thu, 04 Dec 2025 04:11:52 GMT'
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read/analyzeResults/a26d2163-b3ce-4167-b412-73f5bf9746d9?api-version=2023-07-31'
Request method: 'GET'
Request headers:
    'x-ms-client-request-id': '5cfc0af8-d0c7-11f0-b983-56107ba8ade

   ✓ Complete - Decision: conditional_approval

📊 WORKFLOW COMPARISON COMPLETE


In [11]:
# Visualize comparison results
import pandas as pd

# Create comparison DataFrame
df = pd.DataFrame(results)

print("\n📋 COMPARISON TABLE")
print("="*80)
print(df.to_string(index=False))
print("="*80)

# Create visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Decision Distribution', 'Interest Rates', 
                   'Execution Time', 'Total Cost'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# Decision distribution
decision_counts = df['decision'].value_counts()
fig.add_trace(
    go.Pie(labels=decision_counts.index, values=decision_counts.values, 
           name="Decisions"),
    row=1, col=1
)

# Interest rates
fig.add_trace(
    go.Bar(x=df['name'], y=df['interest_rate'], name="Interest Rate (%)",
           marker_color='lightblue'),
    row=1, col=2
)

# Execution time
fig.add_trace(
    go.Bar(x=df['name'], y=df['duration'], name="Duration (s)",
           marker_color='lightgreen'),
    row=2, col=1
)

# Total cost
fig.add_trace(
    go.Bar(x=df['name'], y=df['cost'], name="Cost ($)",
           marker_color='lightcoral'),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text="Multi-Application Workflow Comparison")
fig.show()

# Summary statistics
print("\n📊 SUMMARY STATISTICS")
print(f"   Total Applications: {len(results)}")
print(f"   Approved: {sum(1 for r in results if r['decision'] == 'approved')}")
print(f"   Rejected: {sum(1 for r in results if r['decision'] == 'rejected')}")
print(f"   Conditional: {sum(1 for r in results if r['decision'] == 'conditional')}")
print(f"   Average Duration: {df['duration'].mean():.2f}s")
print(f"   Average Cost: ${df['cost'].mean():.4f}")
print(f"   Total Cost: ${df['cost'].sum():.4f}")


📋 COMPARISON TABLE
           app_id            name         ssn             decision interest_rate  duration  cost  errors
APP-EXCELLENT-001 Alice Excellent 123-45-6789 conditional_approval          6.75 12.448545 0.062       0
     APP-GOOD-002        Bob Good 234-56-7890 conditional_approval          6.75 18.436330 0.062       0
     APP-FAIR-003      Carol Fair 345-67-8901 conditional_approval          6.75 19.112857 0.062       0
     APP-POOR-004      David Poor 456-78-9012 conditional_approval           6.9 17.866898 0.062       0



📊 SUMMARY STATISTICS
   Total Applications: 4
   Approved: 0
   Rejected: 0
   Conditional: 0
   Average Duration: 16.97s
   Average Cost: $0.0620
   Total Cost: $0.2480


## 11. Key Takeaways

### LangGraph State Machine Pattern

1. **Shared State**: All agents update a single `ApplicationState` TypedDict
2. **Sequential Flow**: document → risk → compliance → decision
3. **Error Routing**: Conditional edges check for errors after each agent
4. **Terminal States**: `complete` (success) or `error` (failure)

### Design Benefits

✅ **Observable**: Every state transition is logged with metrics  
✅ **Debuggable**: State accumulates - you can inspect any point in the workflow  
✅ **Error Tolerant**: Errors are collected, workflow routes to error handler  
✅ **Cost Conscious**: Token usage and costs tracked per agent  
✅ **Extensible**: Easy to add new agents or modify flow  

### Performance Characteristics

- **Typical Duration**: 5-15 seconds per application (varies with document complexity)
- **Typical Cost**: $0.05-$0.10 per application
- **Bottlenecks**: Document Intelligence extraction (I/O bound), GPT-4o calls (API bound)
- **Optimization**: Batch processing, caching, parallel document extraction

### Next Steps

- **Phase 8**: Add MLflow experiment tracking to log all workflow metrics
- **Phase 9**: Optimize costs with caching and batch processing
- **Phase 10**: Polish with comprehensive testing and documentation

---

**🎓 Congratulations!** You've successfully orchestrated a multi-agent loan underwriting system with LangGraph.